1. Introducción

La temporada 2015/16 de la Premier League tuvo como protagonista a Leicester City, que consiguió el título en una de las campañas más sorprendentes de la historia del fútbol.

El objetivo de este proyecto es analizar los datos de aquella temporada para comprender qué factores explican el campeonato del Leicester. Se estudiarán su rendimiento general, la comparación con sus principales competidores, su desempeño como local y visitante, su eficiencia ofensiva y las expectativas de las casas de apuestas.

A partir de estos datos, se buscará identificar qué características hicieron posible que Leicester conquistara el título y qué aspectos diferenciaron su rendimiento del resto de los equipos.

Cargamos el dataset

In [1]:
import numpy as np
import pandas as pd

DATA_PATH = "../data/season-1516_csv.csv"
TEAM = "Leicester"

df = pd.read_csv(DATA_PATH)

df.head()

,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,BbAv<2.5,BbAH,BbAHh,BbMxAHH,BbAvAHH,BbMxAHA,BbAvAHA,PSCH,PSCD,PSCA
0,E0,08/08/15,Bournemouth,Aston Villa,0,1,A,0,0,D,...,1.79,26,-0.5,1.98,1.93,1.99,1.92,1.82,3.88,4.70
1,E0,08/08/15,Chelsea,Swansea,2,2,D,2,1,H,...,1.99,27,-1.5,2.24,2.16,1.80,1.73,1.37,5.04,10.88
2,E0,08/08/15,Everton,Watford,2,2,D,0,1,A,...,1.96,26,-1.0,2.28,2.18,1.76,1.71,1.75,3.76,5.44
3,E0,08/08/15,Leicester,Sunderland,4,2,H,3,0,H,...,1.67,26,-0.5,2.00,1.95,1.96,1.90,1.79,3.74,5.10
4,E0,08/08/15,Man United,Tottenham,1,0,H,1,0,H,...,2.01,26,-1.0,2.20,2.09,1.82,1.78,1.64,4.07,6.04


Renombramos las columnas con nombres descriptivos

In [2]:
columnas = {
    "Div": "division",
    "Date": "match_date",
    "HomeTeam": "home_team",
    "AwayTeam": "away_team",
    "FTHG": "home_goals",
    "FTAG": "away_goals",
    "FTR": "full_time_result",
    "HTHG": "home_half_time_goals",
    "HTAG": "away_half_time_goals",
    "HTR": "half_time_result",
    "Referee": "referee",
    "HS": "home_shots",
    "AS": "away_shots",
    "HST": "home_shots_on_target",
    "AST": "away_shots_on_target",
    "HF": "home_fouls",
    "AF": "away_fouls",
    "HC": "home_corners",
    "AC": "away_corners",
    "HY": "home_yellow_cards",
    "AY": "away_yellow_cards",
    "HR": "home_red_cards",
    "AR": "away_red_cards"
}

df = df.rename(columns=columnas)


Exploración inicial

Valores nulos

In [3]:
df.isnull().sum().sort_values(ascending=False)

BWH      1
BWA      1
BWD      1
VCH      0
BbAvH    0
        ..
B365A    0
IWH      0
IWD      0
IWA      0
PSCA     0
Length: 65, dtype: int64

Compruebo que filas tienen los valores nulos

In [4]:
df[df[["BWH", "BWD", "BWA"]].isnull().any(axis=1)]

,division,match_date,home_team,away_team,home_goals,away_goals,full_time_result,home_half_time_goals,away_half_time_goals,half_time_result,...,BbAv<2.5,BbAH,BbAHh,BbMxAHH,BbAvAHH,BbMxAHA,BbAvAHA,PSCH,PSCD,PSCA
158,E0,13/12/15,Tottenham,Newcastle,1,2,A,1,0,H,...,2.35,30,-1.5,2.1,2.03,1.88,1.83,1.37,5.54,9.01


Se identificaron tres valores nulos en las columnas BWH, BWD y BWA, correspondientes a las cuotas de apuestas de Betway para el partido Tottenham vs. Newcastle del 13/12/2015.

Se decidió conservar el registro original, ya que los valores faltantes no afectan los análisis de resultados, goles ni rendimiento de los equipos. Para los análisis de apuestas, se tendrán en cuenta únicamente los valores disponibles.

Registros duplicados

In [5]:
print("Filas duplicadas:", df.duplicated().sum())

Filas duplicadas: 0


Comprobación de los resultados de los partidos

In [6]:
df["full_time_result"].value_counts(dropna=False)

full_time_result
H    157
A    116
D    107
Name: count, dtype: int64

Buscamos partidos donde los goles no coinciden con el resultado registrado

In [7]:
df[
    (
        (df["home_goals"] > df["away_goals"]) &
        (df["full_time_result"] != "H")
    ) |
    (
        (df["home_goals"] < df["away_goals"]) &
        (df["full_time_result"] != "A")
    ) |
    (
        (df["home_goals"] == df["away_goals"]) &
        (df["full_time_result"] != "D")
    )
]

,division,match_date,home_team,away_team,home_goals,away_goals,full_time_result,home_half_time_goals,away_half_time_goals,half_time_result,...,BbAv<2.5,BbAH,BbAHh,BbMxAHH,BbAvAHH,BbMxAHA,BbAvAHA,PSCH,PSCD,PSCA


Se verificó que los resultados finales de los 380 partidos coincidan con los goles registrados. No se encontraron inconsistencias entre los goles del equipo local, los del visitante y el resultado final (H, A o D).

Comprobación de valores negativos

In [8]:
columnas_estadisticas = [
    "home_goals",
    "away_goals",
    "home_half_time_goals",
    "away_half_time_goals",
    "home_shots",
    "away_shots",
    "home_shots_on_target",
    "away_shots_on_target",
    "home_fouls",
    "away_fouls",
    "home_corners",
    "away_corners",
    "home_yellow_cards",
    "away_yellow_cards",
    "home_red_cards",
    "away_red_cards"
]

(df[columnas_estadisticas] < 0).sum()

home_goals              0
away_goals              0
home_half_time_goals    0
away_half_time_goals    0
home_shots              0
away_shots              0
home_shots_on_target    0
away_shots_on_target    0
home_fouls              0
away_fouls              0
home_corners            0
away_corners            0
home_yellow_cards       0
away_yellow_cards       0
home_red_cards          0
away_red_cards          0
dtype: int64

Se revisaron las columnas relacionadas con las estadísticas de los partidos, incluyendo goles, tiros, tiros al arco, faltas, córners y tarjetas. No se encontraron valores negativos en ninguna de las variables analizadas.

Por lo tanto, no fue necesario realizar modificaciones en los datos.

Comprobación de coherencia entre tiros y tiros al arco

In [9]:
# Los tiros al arco nunca pueden superar a los tiros totales
df[
    (df["home_shots_on_target"] > df["home_shots"]) |
    (df["away_shots_on_target"] > df["away_shots"])
][[
    "match_date",
    "home_team",
    "away_team",
    "home_shots",
    "home_shots_on_target",
    "away_shots",
    "away_shots_on_target"
]]


,match_date,home_team,away_team,home_shots,home_shots_on_target,away_shots,away_shots_on_target
170,26/12/15,Aston Villa,West Ham,5,8,10,4


Se encontró una inconsistencia: en el partido Aston Villa vs. West Ham del 26/12/2015, Aston Villa registra 5 tiros totales y 8 tiros al arco, algo imposible.

Se decidió conservar el registro sin modificarlo, porque:

- El dataset no permite saber cuál de los dos valores es el incorrecto, por lo que cualquier corrección sería una suposición.
- El partido no involucra a Leicester ni a los equipos utilizados en la comparación de eficiencia ofensiva, por lo que no afecta los resultados del análisis.

Si en el futuro se comparan las estadísticas de tiros de toda la liga, este partido deberá tenerse en cuenta.

Seleccionamos las columnas relevantes para el análisis. Incluimos las cuotas de Bet365 de victoria local (B365H) y visitante (B365A), que se usan en la sección de apuestas.

In [10]:
df_analysis = df[
    [
        "match_date",
        "home_team",
        "away_team",
        "home_goals",
        "away_goals",
        "full_time_result",
        "home_shots",
        "away_shots",
        "home_shots_on_target",
        "away_shots_on_target",
        "home_corners",
        "away_corners",
        "home_yellow_cards",
        "away_yellow_cards",
        "B365H",
        "B365A"
    ]
]

Transformamos el dataset a formato largo: una fila por equipo por partido.

El dataset original tiene una fila por partido, con columnas separadas para el local (`home_*`) y el visitante (`away_*`). Como todas las preguntas del análisis son por equipo, convertimos cada partido en dos filas: una desde el punto de vista del local y otra desde el visitante. Así, cualquier métrica por equipo se calcula con un `groupby`, sin repetir la lógica de "si es local tomo una columna, si es visitante tomo la otra".

In [11]:
home_columns = {
    "home_team": "team",
    "away_team": "opponent",
    "home_goals": "goals_for",
    "away_goals": "goals_against",
    "home_shots": "shots",
    "home_shots_on_target": "shots_on_target",
    "B365H": "odds",
    "B365A": "opponent_odds"
}

away_columns = {
    "away_team": "team",
    "home_team": "opponent",
    "away_goals": "goals_for",
    "home_goals": "goals_against",
    "away_shots": "shots",
    "away_shots_on_target": "shots_on_target",
    "B365A": "odds",
    "B365H": "opponent_odds"
}

long_columns = [
    "match_date",
    "team",
    "opponent",
    "venue",
    "goals_for",
    "goals_against",
    "shots",
    "shots_on_target",
    "odds",
    "opponent_odds"
]

home = df_analysis.rename(columns=home_columns).assign(venue="home")[long_columns]
away = df_analysis.rename(columns=away_columns).assign(venue="away")[long_columns]

team_matches = pd.concat([home, away], ignore_index=True)

team_matches.head()

,match_date,team,opponent,venue,goals_for,goals_against,shots,shots_on_target,odds,opponent_odds
0,08/08/15,Bournemouth,Aston Villa,home,0,1,11,2,2.00,4.00
1,08/08/15,Chelsea,Swansea,home,2,2,11,3,1.36,11.00
2,08/08/15,Everton,Watford,home,2,2,10,5,1.70,5.50
3,08/08/15,Leicester,Sunderland,home,4,2,19,8,1.95,4.33
4,08/08/15,Man United,Tottenham,home,1,0,9,1,1.65,6.00


El resultado se calcula a partir de los goles. En la sección de validaciones se comprobó que los goles coinciden con `full_time_result` en los 380 partidos, por lo que ambos enfoques son equivalentes.

In [12]:
# Resultado y puntos desde el punto de vista de cada equipo
team_matches["result"] = np.select(
    [
        team_matches["goals_for"] > team_matches["goals_against"],
        team_matches["goals_for"] == team_matches["goals_against"]
    ],
    ["Win", "Draw"],
    default="Loss"
)

team_matches["points"] = team_matches["result"].map({"Win": 3, "Draw": 1, "Loss": 0})

# Validación: 380 partidos x 2 equipos = 760 filas, y 38 partidos por equipo
print("Filas:", len(team_matches))
print("Partidos por equipo:", team_matches["team"].value_counts().unique())

Filas: 760
Partidos por equipo: [38]


Filtramos los partidos de Leicester

In [13]:
leicester = team_matches[team_matches["team"] == TEAM].copy()

leicester.head()

,match_date,team,opponent,venue,goals_for,goals_against,shots,shots_on_target,odds,opponent_odds,result,points
3,08/08/15,Leicester,Sunderland,home,4,2,19,8,1.95,4.33,Win,3
21,22/08/15,Leicester,Tottenham,home,1,1,13,2,2.63,2.75,Draw,1
47,13/09/15,Leicester,Aston Villa,home,3,2,21,6,1.91,4.33,Win,3
60,26/09/15,Leicester,Arsenal,home,2,5,16,7,4.50,1.80,Loss,0
92,24/10/15,Leicester,Crystal Palace,home,1,0,12,3,2.20,3.40,Win,3


2. Rendimiento general del Leicester

Resumen de la temporada

In [14]:
results = leicester["result"].value_counts()

print("Victorias:", results.get("Win", 0))
print("Empates:", results.get("Draw", 0))
print("Derrotas:", results.get("Loss", 0))
print("Puntos Totales:", leicester["points"].sum())
print("Goles a favor:", leicester["goals_for"].sum())
print("Goles en contra:", leicester["goals_against"].sum())

Victorias: 23
Empates: 12
Derrotas: 3
Puntos Totales: 81
Goles a favor: 68
Goles en contra: 36


Conclusión:

El Leicester City finalizó la temporada 2015/16 con 81 puntos, producto de 23 victorias, 12 empates y solo 3 derrotas: no perdió en 35 de los 38 partidos.

Marcó 68 goles y recibió 36, con una diferencia de gol de +32. Los 12 empates aportaron 12 puntos, es decir que incluso cuando no ganaba, solía sumar.

Estos números reflejan un equipo muy regular. En la siguiente sección se compara su rendimiento con el resto de la liga para evaluar qué tan diferencial fue.

3. Comparación con los principales competidores

Creamos una tabla de posiciones completa de la Premier League 2015/16 para comparar el rendimiento del Leicester con otros equipos.

In [15]:
def count_results(result, value):
    return (result == value).sum()

table = team_matches.groupby("team").agg(
    points=("points", "sum"),
    wins=("result", lambda r: count_results(r, "Win")),
    draws=("result", lambda r: count_results(r, "Draw")),
    losses=("result", lambda r: count_results(r, "Loss")),
    goals_for=("goals_for", "sum"),
    goals_against=("goals_against", "sum")
)

table["goal_difference"] = table["goals_for"] - table["goals_against"]

Primer hallazgo importante:

Leicester ganó la liga con 10 puntos de ventaja sobre Arsenal, y fue el equipo que menos partidos perdió: 3, contra 6 de Tottenham, que fue el siguiente.

In [16]:
table = (
    table
    .sort_values(by=["points", "goal_difference"], ascending=False)
    .reset_index()
)
table.insert(0, "position", table.index + 1)

print(table.to_string(index=False))

 position           team  points  wins  draws  losses  goals_for  goals_against  goal_difference
        1      Leicester      81    23     12       3         68             36               32
        2        Arsenal      71    20     11       7         65             36               29
        3      Tottenham      70    19     13       6         69             35               34
        4       Man City      66    19      9      10         71             41               30
        5     Man United      66    19      9      10         49             35               14
        6    Southampton      63    18      9      11         59             41               18
        7       West Ham      62    16     14       8         65             51               14
        8      Liverpool      60    16     12      10         63             50               13
        9          Stoke      51    14      9      15         41             55              -14
       10        Chelsea      

Segundo hallazgo:

Leicester NO fue el equipo más goleador. Si bien quedó a 3 goles de igualar al Manchester City, no se puede decir que fue campeón gracias a su cantidad de goles.


In [17]:
attack_ranking = table.sort_values(by="goals_for", ascending=False)
attack_ranking = attack_ranking[["position", "team", "goals_for"]]

print(attack_ranking.head(10).to_string(index=False))

 position        team  goals_for
        4    Man City         71
        3   Tottenham         69
        1   Leicester         68
        2     Arsenal         65
        7    West Ham         65
        8   Liverpool         63
       10     Chelsea         59
       11     Everton         59
        6 Southampton         59
        5  Man United         49


Tercer hallazgo:

Leicester no fue la mejor defensa: recibió 36 goles, igual que Arsenal, mientras que Tottenham y Manchester United recibieron 35. Aun así, estuvo entre las cuatro defensas menos goleadas, claramente separado del quinto puesto (Manchester City y Southampton, con 41).

In [18]:
defense_ranking = table.sort_values(by="goals_against")
defense_ranking = defense_ranking[["position", "team", "goals_against"]]

print(defense_ranking.to_string(index=False))

 position           team  goals_against
        3      Tottenham             35
        5     Man United             35
        1      Leicester             36
        2        Arsenal             36
        4       Man City             41
        6    Southampton             41
       14      West Brom             48
        8      Liverpool             50
       13        Watford             50
        7       West Ham             51
       15 Crystal Palace             51
       12        Swansea             52
       10        Chelsea             53
        9          Stoke             55
       11        Everton             55
       17     Sunderland             62
       18      Newcastle             65
       19        Norwich             67
       16    Bournemouth             67
       20    Aston Villa             76


Conclusión:

Leicester no lideró ni la tabla de goles a favor ni la de goles recibidos, pero estuvo entre los mejores en ambas: fue el tercer ataque (68 goles) y una de las cuatro defensas menos goleadas (36).

El dato más diferencial es la cantidad de derrotas: perdió solo 3 de 38 partidos, la menor cifra de la liga, mientras que los equipos del 2° al 5° puesto perdieron entre 6 y 10. Esto sugiere que la clave no fue dominar una métrica específica, sino sostener un rendimiento alto y parejo durante toda la temporada.

4. ¿Fortaleza de local o de visitante?

Agrupamos los partidos de Leicester según si jugó de local o de visitante

In [19]:
venue_stats = leicester.groupby("venue").agg(
    matches=("points", "size"),
    points=("points", "sum"),
    wins=("result", lambda r: count_results(r, "Win")),
    goals_for=("goals_for", "sum"),
    goals_against=("goals_against", "sum")
)

venue_stats = venue_stats.loc[["home", "away"]]

venue_stats

,matches,points,wins,goals_for,goals_against
venue,,,,,
home,19,42,12,35,18
away,19,39,11,33,18


Conclusión:

Uno de los aspectos más destacados de la temporada 2015/16 fue la consistencia del Leicester City tanto en casa como fuera de ella. El equipo obtuvo 42 puntos como local y 39 como visitante, mostrando un rendimiento prácticamente idéntico en ambos escenarios.

Además, mantuvo el mismo nivel defensivo al recibir únicamente 18 goles tanto en condición de local como de visitante. En ataque también presentó cifras muy similares, con 35 goles anotados en casa y 33 fuera de ella.

Estos resultados indican que Leicester rindió de forma pareja como local y como visitante, sin depender de jugar en su estadio.

5. Efectividad de cara al arco

Totales de goles, tiros y tiros al arco

In [20]:
print("Goles:", leicester["goals_for"].sum())
print("Tiros:", leicester["shots"].sum())
print("Tiros a puerta:", leicester["shots_on_target"].sum())

Goles: 68
Tiros: 523
Tiros a puerta: 181


Métricas de eficiencia:

Conversión de tiros, conversión de tiros al arco y precisión de tiro cada 100 tiros, y tiros por partido.

In [21]:
conversion_rate = (
    leicester["goals_for"].sum()
    / leicester["shots"].sum()
)

print("Tasa de Conversión:", round(conversion_rate * 100, 2), "%")


shot_on_target_conversion = (
    leicester["goals_for"].sum()
    / leicester["shots_on_target"].sum()
)

print("Conversión de tiros al arco:", round(shot_on_target_conversion * 100, 2), "%")


accuracy = (
    leicester["shots_on_target"].sum()
    / leicester["shots"].sum()
)

print("Precisión:", round(accuracy * 100, 2), "%")

shots_per_match = leicester["shots"].sum() / len(leicester)
print("Tiros por Partido:", round(shots_per_match, 2))

Tasa de Conversión: 13.0 %
Conversión de tiros al arco: 37.57 %
Precisión: 34.61 %
Tiros por Partido: 13.76


Todavía no podemos afirmar si Leicester fue extremadamente eficiente. ¿13.0 % de tasa de conversión es mucho o poco para la Premier League 2015/16?

Vamos a responderlo comparándolo con sus perseguidores.

In [22]:
efficiency = team_matches.groupby("team").agg(
    goals=("goals_for", "sum"),
    shots=("shots", "sum"),
    shots_on_target=("shots_on_target", "sum")
)

efficiency["conversion_rate"] = (efficiency["goals"] / efficiency["shots"] * 100).round(2)
efficiency["accuracy"] = (efficiency["shots_on_target"] / efficiency["shots"] * 100).round(2)
efficiency["shot_on_target_conversion"] = (efficiency["goals"] / efficiency["shots_on_target"] * 100).round(2)

In [23]:
contenders = [
    "Leicester",
    "Arsenal",
    "Tottenham",
    "Man City",
    "Man United"
]

comparison = efficiency.loc[contenders].reset_index()

comparison

,team,goals,shots,shots_on_target,conversion_rate,accuracy,shot_on_target_conversion
0,Leicester,68,523,181,13.00,34.61,37.57
1,Arsenal,65,572,210,11.36,36.71,30.95
2,Tottenham,69,661,250,10.44,37.82,27.60
3,Man City,71,612,210,11.60,34.31,33.81
4,Man United,49,430,143,11.40,33.26,34.27


Conclusión:

El análisis de eficiencia ofensiva revela uno de los factores clave detrás del título del Leicester City. Aunque no fue el equipo que más remató ni el que más ocasiones generó, sí fue el más eficiente de los principales candidatos al campeonato.

Leicester convirtió el 13% de sus tiros en goles, la mejor tasa entre los equipos analizados. Además, el 37.57% de sus tiros al arco terminaron en gol, superando a los otros cuatro equipos.

La comparación con Tottenham resulta especialmente interesante. Mientras que Tottenham registró 661 tiros y 250 tiros al arco, las cifras más altas entre los equipos analizados, Leicester produjo solo 523 tiros y 181 tiros al arco. Sin embargo, ambos equipos terminaron la temporada con una cantidad de goles muy similar (69 para Tottenham y 68 para Leicester). Es decir, Leicester necesitó muchas menos oportunidades para alcanzar prácticamente la misma producción ofensiva.

Estos resultados sugieren que el éxito de Leicester no se basó en un mayor volumen de ataques, sino en aprovechar mejor sus oportunidades. Conviene tomarlo con cautela: el dataset no mide la calidad de las ocasiones (xG), por lo que una conversión alta en una sola temporada puede reflejar tanto calidad de definición como variabilidad.

6. ¿Era favoríto para las casas de apuestas?

Primero creamos una columna que indique si Leicester era favorito según Bet365: es favorito cuando su cuota de victoria es menor que la de su rival.

In [24]:
leicester["favorite"] = leicester["odds"] < leicester["opponent_odds"]

leicester["favorite"].value_counts()

favorite
True     21
False    17
Name: count, dtype: int64

Calculamos victorias y puntos siendo no favorito

In [25]:
underdog = leicester[~leicester["favorite"]]

print("Victorias como no favorito:", count_results(underdog["result"], "Win"))
print("Puntos como no favorito:", underdog["points"].sum())

Victorias como no favorito: 7
Puntos como no favorito: 28


Conclusión:

El análisis de las cuotas de apuestas muestra que Leicester City desafió constantemente las expectativas del mercado durante la temporada 2015/16. A pesar de terminar como campeón, fue considerado no favorito en 17 de los 38 partidos disputados, es decir, en casi la mitad de la temporada.

En esos encuentros logró obtener 28 puntos y 7 victorias, lo que representa aproximadamente el 34.6% de los puntos totales que consiguió en la campaña. Estos resultados evidencian que Leicester no solo fue un equipo eficiente y consistente, sino también uno que superó repetidamente las probabilidades establecidas por las casas de apuestas.

En conjunto, los datos confirman que el título de Leicester City fue una de las mayores sorpresas en la historia reciente de la Premier League.

Conclusión final:

El objetivo de este proyecto fue entender cómo Leicester City ganó la Premier League 2015/16. El análisis identifica varios factores que, en conjunto, explican su campeonato.

En primer lugar, la regularidad. Leicester terminó con 81 puntos (23 victorias, 12 empates y 3 derrotas) y fue el equipo que menos partidos perdió en la liga. Los equipos del 2° al 5° puesto perdieron entre 6 y 10.

En segundo lugar, el equilibrio. No fue el equipo más goleador ni el que menos goles recibió, pero estuvo entre los mejores en ambas facetas: fue el tercer ataque de la liga (68 goles) y una de las cuatro defensas menos goleadas (36).

En tercer lugar, un rendimiento parejo como local y como visitante: sumó 42 puntos en casa y 39 fuera, y recibió 18 goles en cada condición.

En cuarto lugar, la eficiencia ofensiva. Tuvo la mejor tasa de conversión entre los cinco primeros (13% de sus tiros terminaron en gol) y, con 138 tiros menos que Tottenham, marcó solo un gol menos. Como el dataset no incluye la calidad de las ocasiones, no es posible determinar cuánto de esa eficiencia se debe a la definición y cuánto a la variabilidad.

Finalmente, el análisis de las cuotas de apuestas confirmó la magnitud de la sorpresa. Leicester fue considerado no favorito en 17 de los 38 partidos disputados y obtuvo 28 puntos en esos encuentros, superando de manera constante las expectativas del mercado. Incluso durante gran parte de la temporada, las casas de apuestas continuaron mostrando cierto escepticismo respecto a sus posibilidades reales de ser campeón.

En conclusión, los datos indican que Leicester City no ganó la Premier League 2015/16 por dominar una estadística específica, sino por combinar regularidad, equilibrio y eficiencia.